In [ ]:
import glob
import json

# Initialize an empty dictionary to hold the merged mesh_signatures
merged_mesh_fingerprints = {}

# Get the list of files matching the pattern
file_pattern = r'D:\BEHAVIOR-1K\asset_pipeline\cad\*\*\artifacts\object_list.json'
files = glob.glob(file_pattern)

# Iterate over each file and merge the mesh_signatures dictionaries
for file in files:
  with open(file, 'r') as f:
    data = json.load(f)
    if 'mesh_fingerprints' in data:
      merged_mesh_fingerprints.update(data['mesh_fingerprints'])

# Now merged_mesh_signatures contains the merged dictionary
print(merged_mesh_fingerprints)

In [ ]:
# Get the moment of inertia for each mesh
import numpy as np
n = len(merged_mesh_fingerprints)
moments_of_inertia = np.stack([np.array(x[2]).reshape(3, 3)[np.triu_indices(3)] for x in merged_mesh_fingerprints.values()])
print(moments_of_inertia.shape)

In [ ]:
# For each of the 9 elements in the moment of inertia, get the mean and standard deviation
means = np.mean(moments_of_inertia, axis=0)
std_devs = np.std(moments_of_inertia, axis=0)

print(means)
print(std_devs)

In [ ]:
# Compute sample covariance
covariance = np.cov(moments_of_inertia, rowvar=False, bias=True)
inv_covariance = np.linalg.inv(covariance)
print(covariance)

In [ ]:
# Compute mahalanobis distances between every pair of meshes
def pairwise_mahalanobis(V, SI):
    """
    Compute pairwise Mahalanobis distances between vectors in V using inverse covariance matrix SI.
    
    Parameters:
    V: numpy array of shape (n, p) where n is number of vectors and p is dimensionality
    SI: inverse covariance matrix of shape (p, p)
    
    Returns:
    D: numpy array of shape (n, n) containing pairwise Mahalanobis distances
    """
    # Get number of vectors
    n = V.shape[0]
    
    # Compute difference vectors for all pairs
    # Using broadcasting to create a (n, n, p) array
    diff = V[:, np.newaxis, :] - V  # Shape: (n, n, p)
    
    # Compute Mahalanobis distances
    # For each pair, compute diff @ SI @ diff.T
    # Using einsum for efficient computation
    D = np.sqrt(np.einsum('ijk,kl,ijl->ij', diff, SI, diff))
    
    return D

# Compute pairwise Mahalanobis distances
D = pairwise_mahalanobis(moments_of_inertia, inv_covariance)

In [ ]:
D.shape

In [ ]:
different_object_distances = D[~np.eye(D.shape[0],dtype=bool)]
different_object_distances = different_object_distances[~np.isnan(different_object_distances)]

In [ ]:
# Describe the distribution of Mahalanobis distances
print(np.median(different_object_distances))
print(np.mean(different_object_distances))
print(np.std(different_object_distances))
print(np.min(different_object_distances))
print(np.max(different_object_distances))
print(np.percentile(different_object_distances, 25))
print(np.percentile(different_object_distances, 50))
print(np.percentile(different_object_distances, 75))

In [ ]:
# Plot a histogram of the 10-to-90th percentile of Mahalanobis distances
import matplotlib.pyplot as plt
min_val = np.percentile(different_object_distances, 10)
max_val = np.percentile(different_object_distances, 90)
plt.hist(different_object_distances, bins=100, range=(min_val, max_val))
plt.show()

In [ ]:
# Lets look at a particular example, jatssq
bad = (13014, 10432, [0.11338169027658968, -0.0006982750120267411, -0.001630522249463943, -0.0006982750120267411, 0.11209346603884637, -0.0015877659633802192, -0.001630522249463943, -0.0015877659633802192, 0.08144469651441441], [-0.04741873350956649, -0.08014451253401145, -0.233840203355693])
good = merged_mesh_fingerprints["jatssq"]

# Replace both MOIs with the upper triangles
bad_upper = np.array(bad[2]).reshape(3, 3)[np.triu_indices(3)]
good_upper = np.array(good[2]).reshape(3, 3)[np.triu_indices(3)]

print(bad)
print(good)
print(bad_upper - good_upper)

# Compute the Mahalanobis distance between the two
diff = bad_upper - good_upper
mahalanobis = np.sqrt(diff @ inv_covariance @ diff)
print(mahalanobis)M